# Phylogenetic Tree Recovery — Diagnostic Run

End-to-end runner for `rep-phylogeny`'s diagnostic suite (`diagnostics.md`).

**What it does**
1. Extracts 4 strategic layers (~quarter/half/three-quarter/final) for
   `xlm-roberta-large` (24 total, fp16, mean-pool) and `google/gemma-2-2b`
   (26 total, fp32, mean-pool).
2. Runs Blocks A–H of the diagnostic suite for each `(model, pool, layer) ×
   (procrustes, ridge)` configuration — 16 configs total.
3. Writes per-config logs and `SUMMARY.txt`. The summary is rewritten after
   every config, so a crash leaves a usable artifact.

**Setup**
- Accelerator: **GPU T4 x2**.
- Add a Kaggle Secret named `HF_TOKEN` and accept the Gemma-2 license.
- Expected wall time: ≈45 min extraction + ≈1.5–2 h diagnostics ≈ **2.5–3 h**.

## 1. Clone repo and install deps

In [ ]:
import os, subprocess
WORKDIR = "/kaggle/working"
REPO_DIR = os.path.join(WORKDIR, "rep-phylogeny")
os.chdir(WORKDIR)
if os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--rebase"])
else:
    subprocess.check_call(["git", "clone", "https://github.com/saketatreya/rep-phylogeny.git", REPO_DIR])
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!pip install -q -r requirements.txt

## 2. HuggingFace token (required for Gemma-2-2b)

Add a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) and accept the
[Gemma-2 license](https://huggingface.co/google/gemma-2-2b). Without this,
run only XLM-R by passing `--models xlm-roberta-large` to the cell below.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF token loaded.")
except Exception as e:
    print(f"No HF token: {e}")
    print("Gemma-2 will fail. Run only XLM-R below by passing --models xlm-roberta-large.")

## 3. Run diagnostics — strategic 4-layer probe per model

Both models, both methods (Procrustes + Ridge), Blocks A–H, mean_pool only.
Layers sampled at ~quarter/half/three-quarter/final depth:

- XLM-R (24 layers): `layer_05`, `layer_11`, `layer_17`, `layer_23`
- Gemma (26 layers): `layer_06`, `layer_12`, `layer_18`, `layer_25`

The driver intersects `--layers` with each model's available labels, so passing
all 8 in one list is fine. `--n-perms 5` keeps Block D's z-scores readable
while cutting the most expensive block roughly in half. Expected wall time:
≈45 min extraction + ≈1.5–2 h diagnostics ≈ **2.5–3 h** total. `SUMMARY.txt`
is rewritten after every config so a crash leaves a usable artifact.

In [ ]:
!python run_diagnostics.py \
    --out-dir /kaggle/working/results_diag \
    --pools mean_pool \
    --layers layer_05 layer_11 layer_17 layer_23 layer_06 layer_12 layer_18 layer_25 \
    --n-perms 5

## 4. Inspect the master summary

In [ ]:
print(open("/kaggle/working/results_diag/SUMMARY.txt").read())

## 5. (Optional) drill into a single config

Per-config logs live at `results_diag/{model}/{pool}/{layer}/{method}.log`
and `block_a.log`. The `sanity.log` at `results_diag/{model}/{pool}/` lists
norms and Spa-Por/Fre/Ron cosines for every layer.

In [ ]:
# Example: XLM-R, mean_pool, layer_12, ridge
p = "/kaggle/working/results_diag/xlm-roberta-large/mean_pool/layer_12/ridge.log"
print(open(p).read())